# Alzheimer's Disease Classification
End-to-end ML pipeline comparing Logistic Regression, Random Forest, and XGBoost.

In [ ]:
import os
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# RANDOM_STATE fixes the seed so results are identical every run
RANDOM_STATE = 42
FIGURES_DIR  = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Load and Explore Data

In [ ]:
df = pd.read_csv('data/alzheimers_disease_data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Shows each column's data type and non-null count — how we spot missing values
df.info()

In [ ]:
# include='all' covers text columns too; .T flips rows/cols for readability
df.describe(include='all').T

In [ ]:
# 0 = healthy, 1 = Alzheimer's — if very unbalanced the model may be biased
print('Diagnosis distribution:')
print(df['Diagnosis'].value_counts(normalize=True).round(3))

## 2. Preprocessing

In [ ]:
# PatientID is just a row number; DoctorInCharge is identical for every patient
DROP_COLS = ['PatientID', 'DoctorInCharge']
df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

y = df_clean['Diagnosis'].astype(int)
X = df_clean.drop(columns=['Diagnosis'])

# Convert categorical columns to 0/1 — models can't work with raw text
# drop_first=True removes one redundant dummy per category (avoids multicollinearity)
X = pd.get_dummies(X, drop_first=True)

print('Number of features:', X.shape[1])

In [ ]:
# stratify=y preserves the 0/1 ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Scale so Logistic Regression isn't dominated by large-valued features
# fit only on train data — applying fit to test would be data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Train target distribution:', y_train.value_counts(normalize=True).round(3).to_dict())

## 3. Model Definitions

In [ ]:
# Logistic Regression needs scaled input; tree-based models do not
SCALED_MODELS = {'Logistic Regression'}

models = {
    # Simple linear baseline — if others can't beat this, something is wrong
    'Logistic Regression': LogisticRegression(
        max_iter=1000,        # default 100 sometimes isn't enough to converge
        random_state=RANDOM_STATE,
    ),
    # Builds 200 trees and combines their votes
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,            # use all CPU cores
    ),
    # Adds trees one by one; each new tree corrects the previous one's mistakes
    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=6,          # limits depth to avoid overfitting
        learning_rate=0.1,    # how much each new tree contributes
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

print('Models:', list(models.keys()))

## 4. Training and Prediction

In [ ]:
fitted      = {}
predictions = {}

for name, model in models.items():
    # Logistic Regression gets scaled data; tree models get the original features
    X_tr = X_train_scaled if name in SCALED_MODELS else X_train
    X_te = X_test_scaled  if name in SCALED_MODELS else X_test

    model.fit(X_tr, y_train)
    predictions[name] = model.predict(X_te)
    fitted[name]      = model

    print(f'{name}: training complete')

## 5. Evaluation

In [ ]:
accuracies = {name: accuracy_score(y_test, y_pred) for name, y_pred in predictions.items()}

for name, acc in accuracies.items():
    print(f'{name}: {acc:.4f}')

In [ ]:
# precision: of all predicted Yes, how many were right
# recall:    of all actual Yes, how many did we catch
# f1-score:  balanced combination of both
for name, y_pred in predictions.items():
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))

In [ ]:
# top-right = false positives, bottom-left = false negatives (missed Alzheimer's cases)
fig, axes = plt.subplots(1, len(predictions), figsize=(5 * len(predictions), 4))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'], ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_matrices.png', dpi=120)
plt.show()

In [ ]:
# Random Forest provides feature_importances_ directly after training
# Sort ascending + tail so the most important feature appears at the top of the chart
importances = (
    pd.Series(fitted['Random Forest'].feature_importances_, index=X.columns)
    .sort_values(ascending=True)
    .tail(15)
)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/feature_importance.png', dpi=120)
plt.show()

In [ ]:
names  = list(accuracies.keys())
scores = list(accuracies.values())

plt.figure(figsize=(7, 4))
bars = plt.bar(names, scores, color=['#4C72B0', '#55A868', '#C44E52'])

# Start y-axis just below the lowest score so small differences are visible
plt.ylim(max(0.0, min(scores) - 0.05), 1.0)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')

for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width() / 2, score + 0.005,
             f'{score:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/accuracy_comparison.png', dpi=120)
plt.show()

## 6. Summary

In [ ]:
# Baseline = always predicting the majority class; any real model must beat this
baseline   = y_test.value_counts(normalize=True).max()
best_model = max(accuracies, key=accuracies.get)

print('=== Summary ===')
print(f'Baseline (majority class): {baseline:.4f}')
for name, acc in accuracies.items():
    print(f'{name:22s}  accuracy={acc:.4f}  (+{acc - baseline:.4f} vs baseline)')

print(f'\nBest model: {best_model} ({accuracies[best_model]:.4f})')

# Top 5 features — tells us which patient attributes matter most for the diagnosis
print('\nTop 5 features (Random Forest):')
print(importances.tail(5)[::-1].round(4).to_string())

print('\nFigures saved to figures/')